In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import SPSA

# Simulator without noise that returns shots (replaces the ibmq_jakarta hardware)
sampler = StatevectorSampler(seed=42)
SHOTS = 1024

In [ ]:
from py4lexis.session import LexisSession
from qaas.client import QProvider, QBackend, QJob

# Login to LEXIS
lexis_session = LexisSession()
token = lexis_session.get_access_token()

# VLQ project and resource
PROJECT = "vlq_deic_dk_demo_project"
RESOURCE = "VLQ-DK"

# Initialize QaaS
provider = QProvider(PROJECT, token)

backend: QBackend = provider.get_backend(RESOURCE)
pulla = provider.get_pulla(RESOURCE)

print("Successfully connected to VLQ!")
print("Gate-level backend:", backend)
print("Pulse-level Pulla:", pulla)

In [ ]:
def build_circuit(x, n, theta, measure=True):   # n = number of qubits = number of features
    qc = QuantumCircuit(n)

    # Feature map, 2 layers (Fig. 1b)
    for _ in range(2):
        for i in range(n):
            qc.h(i)
            qc.rz(2 * x[i], i)
        for i in range(n - 1):          # RZZ on neighbouring pairs (0,1), (1,2), ...
            x_ij = 2 * (np.pi - x[i]) * (np.pi - x[i + 1])
            qc.rzz(x_ij, i, i + 1)

    # Ansatz (Fig. 1a)
    for i in range(n):
        qc.ry(theta[i], i)              # first row: theta[0] ... theta[n-1]
    for i in range(n - 1):
        qc.cx(i, i + 1)                 # CNOT chain
    for i in range(n):
        qc.ry(theta[n + i], i)          # second row: theta[n] ... theta[2n-1]

    if measure:
        qc.measure_all()
    qc_transpiled = transpile_to_IQM(qc, backend)
    return qc_transpiled

In [ ]:
def p_class0(X, n, theta, shots=SHOTS):
    circuits = [build_circuit(x, n, theta, measure=True) for x in X]
    results = sampler.run(circuits, shots=shots).result()

    p0 = []
    for r in results:
        counts = r.data.meas.get_counts()
        # even parity (even number of 1s) -> class 0
        even = sum(c for bits, c in counts.items() if bits.count("1") % 2 == 0)
        p0.append(even / shots)
    return np.array(p0)


def model(x, n, theta):
    # exact parity <Z⊗Z⊗...⊗Z> (Eq. 1), used only to generate the dataset
    qc = build_circuit(x, n, theta, measure=False)
    P = SparsePauliOp("Z" * n)
    return Statevector(qc).expectation_value(P).real

In [ ]:
def points(X_pool, n, theta_s):
    m_pool = np.array([model(x, n, theta_s) for x in X_pool])
    order = np.argsort(m_pool)             # sort most negative to most positive
    idx_class1 = order[:50]                # m < 0, odd parity -> class 1
    idx_class0 = order[-50:]               # m > 0, even parity   -> class 0
    X = np.vstack([X_pool[idx_class0], X_pool[idx_class1]])
    y = np.array([0]*50 + [1]*50)
    return X, y

In [ ]:
def loss(theta, X, y, n): #L(θ) = −(1/N) · Σᵢ log p(yᵢ | xᵢ, θ)
    p0 = p_class0(X, n, theta)
    p0 = np.clip(p0, 1e-10, 1 - 1e-10) #avoid log(0) and log(1)
    p_correct = np.where(y == 0, p0, 1 - p0) #if class 0, p_correct = p0, if class 1, p_correct = 1 - p0
    return -np.mean(np.log(p_correct))

In [ ]:
n_list = [2, 3, 4, 5]    # number of qubits

def case1_gen(n):
    algorithm_globals.random_seed = 0
    rng = np.random.default_rng(seed=0)

    # 1. Dataset (exact simulation)
    X_all = rng.uniform(0, 1, size=(600, n))

    def cost(theta):  # defined here, uses this X_all
        m = np.array([model(x, n, theta) for x in X_all])
        return -np.mean(np.abs(m))

    theta_init = rng.uniform(0, 2 * np.pi, size=2 * n)
    theta_s = minimize(cost, theta_init, method="L-BFGS-B").x
    X_train, y_train = points(X_all, n, theta_s)

    X_all_test = rng.uniform(0, 1, size=(600, n))
    X_test, y_test = points(X_all_test, n, theta_s)


       # 2. Training with Qiskit's SPSA
    theta_0 = rng.uniform(0, 2 * np.pi, size=2 * n)   # 2n parameters

    loss_history = []
    def f(theta):
        value = loss(theta, X_train, y_train, n)
        loss_history.append(value)
        np.save(f"progress_n{n}.npy", {"loss_history": loss_history, "theta": theta}, allow_pickle=True)
        return value

    optimizer = SPSA(maxiter=120)
    result = optimizer.minimize(f, theta_0)
    theta_train = result.x

    # 3. Results for Fig. 3
    final_loss = loss(theta_train, X_train, y_train, n)
    p0_test = p_class0(X_test, n, theta_train)
    y_pred = np.where(p0_test > 0.5, 0, 1)            # class 0 if p0 > 0.5, else class 1
    acc = np.mean(y_pred == y_test)
    np.save(f"results_n{n}.npy", {"loss_history": loss_history, "final_loss": final_loss, "acc": acc, "theta_train": theta_train}, allow_pickle=True)
    return loss_history, final_loss, acc

In [ ]:
losses, accs = [], []
for n in n_list:
    loss_history, final_loss, acc = case1_gen(n)
    losses.append(final_loss)
    accs.append(acc)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(n_list, losses, "x-", label="Noiseless simulation")
ax[0].set_xlabel("Number of qubits"); ax[0].set_ylabel("Training loss"); ax[0].set_title("(a)", loc="left")
ax[1].plot(n_list, accs, "x-", label="Noiseless simulation")
ax[1].set_xlabel("Number of qubits"); ax[1].set_ylabel("Classification acc."); ax[1].set_title("(b)", loc="left")
for a in ax:
    a.set_xticks(n_list)
    a.legend()
plt.tight_layout()
plt.savefig("fig3.jpg", bbox_inches="tight")
plt.show()